# Lab 1 — Multimodal RAG
## Pipeline: Docling → CLIP → OpenSearch → Query Multimodal

**Aula 11 · MBA RAG & CAG Aplicados a Direito e Segurança Pública**

**Objetivo:** Construir um pipeline RAG capaz de recuperar **texto e imagens** de forma unificada.  
Ao final, uma única query textual deve retornar simultaneamente um parágrafo relevante e uma imagem relacionada.

**Duração estimada:** 60 minutos

---

### Arquitetura do Lab

```
PDF Jurídico (simulado)
       │
       ▼
   Docling
    ├── Texto (parágrafos)
    ├── Tabelas (markdown)
    └── Imagens (PNG)
       │
  ┌────┴─────────────────────┐
  ▼                          ▼
BGE-M3                     CLIP
(texto/tabelas)          (imagens)
  │                          │
  └─────────┬────────────────┘
             ▼
    OpenSearch Index
    (campo: embedding 768d)
             │
     Query textual
             │
    ┌────────┴───────────┐
    │   KNN Search       │
    │  type=text         │
    │  type=image        │
    └────────────────────┘
```

## Etapa 0 — Instalação de Dependências

In [ ]:
# Instalar dependências do Lab 1
# Tempo estimado: 3-5 minutos no Colab

!pip install -q docling==2.5.0
!pip install -q transformers==4.41.0 torch torchvision
!pip install -q Pillow==10.3.0
!pip install -q opensearch-py==2.4.2
!pip install -q sentence-transformers==3.0.0
!pip install -q fpdf2==2.7.9  # Para criar PDF de exemplo

print("✅ Dependências instaladas com sucesso!")

## Etapa 1 — Criação do Documento PDF de Exemplo

Neste lab, criamos um PDF simulando um **relatório de inteligência policial** com texto e imagens incorporadas.

In [ ]:
import os
import json
import numpy as np
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
import io

# Criar diretório de trabalho
WORK_DIR = Path("/content/lab1_multimodal")
WORK_DIR.mkdir(exist_ok=True)
(WORK_DIR / "imagens").mkdir(exist_ok=True)
(WORK_DIR / "index_data").mkdir(exist_ok=True)

print(f"📁 Diretório de trabalho: {WORK_DIR}")

In [ ]:
# Criar imagens de exemplo simulando conteúdo forense
# Em produção, estas seriam imagens reais extraídas pelo Docling

def criar_imagem_diagrama(titulo: str, conteudo: list[str], filename: str, cor_fundo: str = "#E8F4F8") -> Path:
    """Cria uma imagem PNG simulando diagrama de documento jurídico."""
    img = Image.new("RGB", (600, 400), color=cor_fundo)
    draw = ImageDraw.Draw(img)
    
    # Título
    draw.rectangle([20, 20, 580, 70], fill="#2C3E50", outline="#1A252F")
    draw.text((30, 35), titulo, fill="white")
    
    # Conteúdo
    y = 90
    for linha in conteudo:
        draw.text((30, y), f"• {linha}", fill="#2C3E50")
        y += 30
    
    # Rodapé
    draw.rectangle([20, 350, 580, 380], fill="#BDC3C7", outline="#95A5A6")
    draw.text((30, 358), "Documento Confidencial - Uso Restrito", fill="#2C3E50")
    
    caminho = WORK_DIR / "imagens" / filename
    img.save(caminho)
    return caminho


# Imagem 1: Organograma de facção (simulado)
img1 = criar_imagem_diagrama(
    "Estrutura Organizacional - Célula Norte",
    [
        "Liderança: 1 indivíduo (codinome 'Falcon')",
        "Subchefes: 3 (distribuição territorial)",
        "Operadores: 12 (tráfico e extorsão)",
        "Área de atuação: Bairros A, B e C",
        "Armamento apreendido: 4 pistolas, 2 rifles",
    ],
    "organograma_celula_norte.png"
)

# Imagem 2: Mapa de incidências criminais (simulado)
img2 = criar_imagem_diagrama(
    "Mapa de Calor - Ocorrências 2024",
    [
        "Alta incidência: Zonas 3, 7 e 12",
        "Período crítico: 22h - 04h",
        "Tipo predominante: Roubo a transeunte (43%)",
        "Segundo tipo: Tráfico de entorpecentes (28%)",
        "Delegacia responsável: 5ª DP",
    ],
    "mapa_calor_ocorrencias.png",
    cor_fundo="#FEF9E7"
)

# Imagem 3: Tabela de apreensões
img3 = criar_imagem_diagrama(
    "Tabela de Apreensões - Operação Corvina",
    [
        "Drogas: 2.3 kg cocaína, 15 kg maconha",
        "Armas: 6 pistolas .40, 2 espingardas",
        "Dinheiro: R$ 47.300 em espécie",
        "Veículos: 3 motos, 1 automóvel",
        "Presos: 8 indivíduos (Art. 33 Lei 11.343/06)",
    ],
    "tabela_apreensoes_operacao.png",
    cor_fundo="#FDEDEC"
)

print(f"✅ Imagens criadas:")
print(f"   📸 {img1}")
print(f"   📸 {img2}")
print(f"   📸 {img3}")

## Etapa 2 — Extração com Docling (Simulação)

Em ambiente de produção, o Docling processa PDFs reais. Aqui simulamos sua saída estruturada com os mesmos campos que o Docling produziria.

In [ ]:
# Simulação da saída do Docling
# Em produção:
#   from docling.document_converter import DocumentConverter
#   converter = DocumentConverter()
#   result = converter.convert("relatorio.pdf")
#   for chunk in result.document.export_to_markdown():
#       ...

# Documentos de texto que o Docling extrairia do PDF
chunks_texto = [
    {
        "id": "txt_001",
        "type": "text",
        "conteudo": """RELATÓRIO DE INTELIGÊNCIA POLICIAL — OPERAÇÃO CORVINA
        Data: 15/03/2024. A Operação Corvina foi deflagrada pela Delegacia 
        Especializada em Crimes Organizados (DECO) com fundamento no 
        Art. 2º da Lei 12.850/2013 (Lei das Organizações Criminosas). 
        O objetivo foi desarticular a célula norte do grupo criminoso 
        identificado pelo codinome interno 'Corvos'.""",
        "pagina": 1,
        "secao": "Introdução"
    },
    {
        "id": "txt_002",
        "type": "text",
        "conteudo": """FUNDAMENTO LEGAL: O procedimento observou o Art. 33 da 
        Lei 11.343/2006 (tráfico de drogas), o Art. 157 do Código Penal 
        (roubo) e o Art. 16 do Estatuto do Desarmamento (Lei 10.826/2003). 
        Os mandados de busca e apreensão foram expedidos nos termos do 
        Art. 240 do Código de Processo Penal, com autorização judicial 
        da 3ª Vara Criminal.""",
        "pagina": 2,
        "secao": "Fundamento Legal"
    },
    {
        "id": "txt_003",
        "type": "text",
        "conteudo": """RESULTADO DA OPERAÇÃO: Foram presos 8 indivíduos em flagrante 
        delito. O principal suspeito, identificado como líder da organização, 
        responde por crime de associação ao tráfico (Art. 35 da Lei 11.343/2006) 
        e liderança de organização criminosa (Art. 2º, §3º da Lei 12.850/2013). 
        As apreensões incluem armas de fogo, entorpecentes e valores em espécie.""",
        "pagina": 3,
        "secao": "Resultados"
    },
    {
        "id": "txt_004",
        "type": "text",
        "conteudo": """ANÁLISE DE INTELIGÊNCIA: A célula norte opera em 3 bairros 
        distintos com hierarquia definida. A estrutura organizacional identificada 
        revela um lider (codinome 'Falcon'), três subchefes com controle territorial 
        e doze operadores. O padrão de comunicação utiliza aplicativos criptografados, 
        dificultando a interceptação telemática prevista no Art. 1º da Lei 9.296/1996.""",
        "pagina": 4,
        "secao": "Análise de Inteligência"
    },
    {
        "id": "txt_005",
        "type": "text",
        "conteudo": """MAPA CRIMINAL: O levantamento estatístico das ocorrências 
        registradas no período de janeiro a março de 2024 indica alta concentração 
        de crimes nas zonas 3, 7 e 12 do município. O horário de maior incidência 
        é entre 22h e 04h. O tipo penal mais frequente é roubo a transeunte (43%), 
        seguido de tráfico de entorpecentes (28%).""",
        "pagina": 5,
        "secao": "Estatísticas"
    },
]

# Imagens que o Docling extrairia do PDF
chunks_imagem = [
    {
        "id": "img_001",
        "type": "image",
        "caminho": str(img1),
        "caption": "Organograma da célula norte da organização criminosa identificada",
        "pagina": 4,
        "secao": "Análise de Inteligência"
    },
    {
        "id": "img_002",
        "type": "image",
        "caminho": str(img2),
        "caption": "Mapa de calor das ocorrências criminais registradas em 2024",
        "pagina": 5,
        "secao": "Estatísticas"
    },
    {
        "id": "img_003",
        "type": "image",
        "caminho": str(img3),
        "caption": "Tabela de materiais apreendidos durante a Operação Corvina",
        "pagina": 3,
        "secao": "Resultados"
    },
]

print(f"📄 Chunks de texto extraídos: {len(chunks_texto)}")
print(f"🖼️  Imagens extraídas: {len(chunks_imagem)}")
print("\nEstrutura de um chunk de texto:")
print(json.dumps({k: v for k, v in chunks_texto[0].items() if k != 'conteudo'}, ensure_ascii=False, indent=2))
print(f"\nConteúdo: {chunks_texto[0]['conteudo'][:100]}...")

## Etapa 3 — Geração de Embeddings

Usamos **BGE-M3** para texto e **CLIP** para imagens. Ambos produzem vetores de dimensão compatível para busca unificada.

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch

print("📥 Carregando modelos de embedding...")

# Modelo para texto
modelo_texto = SentenceTransformer("BAAI/bge-m3")
print("✅ BGE-M3 carregado (para texto e tabelas)")

# Modelo para imagens
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()
print("✅ CLIP ViT-B/32 carregado (para imagens)")

# Dimensões
DIM_TEXTO = 1024   # BGE-M3
DIM_CLIP = 512     # CLIP
print(f"\nDimensão embeddings texto (BGE-M3): {DIM_TEXTO}")
print(f"Dimensão embeddings imagem (CLIP): {DIM_CLIP}")
print("\n⚠️  Nota: Para indexação unificada, usaremos projeção linear para equalizar dimensões")

In [ ]:
import numpy as np
from typing import Literal

def embed_texto(texto: str) -> np.ndarray:
    """Gera embedding de texto com BGE-M3."""
    embedding = modelo_texto.encode(
        texto,
        normalize_embeddings=True,
        show_progress_bar=False
    )
    # Projeção para 512 dimensões (para compatibilidade com CLIP)
    # Em produção: usar PCA ou projeção linear treinada
    return embedding[:512]  # Primeiras 512 dims do BGE-M3


def embed_imagem(caminho_imagem: str) -> np.ndarray:
    """Gera embedding de imagem com CLIP."""
    imagem = Image.open(caminho_imagem).convert("RGB")
    inputs = clip_processor(images=imagem, return_tensors="pt")
    with torch.no_grad():
        features = clip_model.get_image_features(**inputs)
        features = features / features.norm(dim=-1, keepdim=True)  # Normaliza
    return features.squeeze().numpy()


def embed_query_clip(texto: str) -> np.ndarray:
    """Gera embedding de texto com CLIP (para busca de imagens)."""
    inputs = clip_processor(text=[texto], return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        features = clip_model.get_text_features(**inputs)
        features = features / features.norm(dim=-1, keepdim=True)
    return features.squeeze().numpy()


# Gerar embeddings para todos os chunks de texto
print("🔄 Gerando embeddings de texto (BGE-M3)...")
for chunk in chunks_texto:
    chunk["embedding"] = embed_texto(chunk["conteudo"]).tolist()
    print(f"  ✅ {chunk['id']} ({len(chunk['embedding'])} dims)")

print("\n🔄 Gerando embeddings de imagem (CLIP)...")
for chunk in chunks_imagem:
    chunk["embedding"] = embed_imagem(chunk["caminho"]).tolist()
    print(f"  ✅ {chunk['id']} ({len(chunk['embedding'])} dims)")

print("\n✅ Todos os embeddings gerados!")
print(f"Total de documentos indexáveis: {len(chunks_texto) + len(chunks_imagem)}")

## Etapa 4 — Indexação no OpenSearch (Simulação Local)

Por limitações do Colab, usamos uma implementação NumPy que replica o comportamento do KNN Search do OpenSearch.

In [ ]:
# Implementação de KNN Search local (replica OpenSearch KNN)
# Em produção: substituir por cliente opensearch-py com índice knn_vector

class IndiceMultimodal:
    """
    Índice vetorial local que simula o comportamento do OpenSearch KNN.
    
    Configuração equivalente no OpenSearch:
    PUT /indice_multimodal
    {
      "mappings": {
        "properties": {
          "embedding": {
            "type": "knn_vector",
            "dimension": 512,
            "method": {"name": "hnsw", "space_type": "cosinesimil"}
          },
          "tipo": {"type": "keyword"},
          "conteudo": {"type": "text"},
          "pagina": {"type": "integer"}
        }
      }
    }
    """
    
    def __init__(self):
        self.documentos = []
        self.embeddings = []
    
    def indexar(self, documento: dict):
        """Adiciona documento ao índice."""
        embedding = np.array(documento.pop("embedding"))
        self.embeddings.append(embedding)
        self.documentos.append(documento)
    
    def buscar(self, query_embedding: np.ndarray, k: int = 5, tipo_filtro: str = None) -> list[dict]:
        """
        Busca KNN por similaridade coseno.
        
        Equivalente OpenSearch:
        POST /indice_multimodal/_search
        {
          "query": {
            "knn": {
              "embedding": {"vector": query_embedding, "k": k}
            }
          },
          "post_filter": {"term": {"tipo": tipo_filtro}}
        }
        """
        if not self.embeddings:
            return []
        
        # Calcular similaridade coseno com todos os documentos
        matriz = np.array(self.embeddings)
        
        # Normalizar query
        q_norm = query_embedding / (np.linalg.norm(query_embedding) + 1e-9)
        
        # Normalizar documentos
        norms = np.linalg.norm(matriz, axis=1, keepdims=True) + 1e-9
        matriz_norm = matriz / norms
        
        scores = matriz_norm @ q_norm
        
        # Aplicar filtro de tipo
        indices_validos = range(len(self.documentos))
        if tipo_filtro:
            indices_validos = [
                i for i in indices_validos 
                if self.documentos[i].get("type") == tipo_filtro
            ]
        
        # Ordenar por score
        resultados = sorted(
            [(i, scores[i]) for i in indices_validos],
            key=lambda x: x[1],
            reverse=True
        )[:k]
        
        return [
            {**self.documentos[i], "_score": float(s)}
            for i, s in resultados
        ]


# Inicializar e popular o índice
indice = IndiceMultimodal()

# Indexar documentos de texto
for chunk in chunks_texto:
    # Precisamos regenerar o embedding pois pop() foi chamado acima
    chunk_copia = chunk.copy()
    indice.indexar(chunk_copia)

# Indexar imagens
for chunk in chunks_imagem:
    chunk_copia = chunk.copy()
    indice.indexar(chunk_copia)

# Reindexar com embeddings corretos
indice = IndiceMultimodal()

for chunk in chunks_texto:
    doc = {k: v for k, v in chunk.items() if k != 'embedding'}
    doc["embedding"] = chunk["embedding"]
    indice.indexar(doc)

for chunk in chunks_imagem:
    doc = {k: v for k, v in chunk.items() if k != 'embedding'}
    doc["embedding"] = chunk["embedding"]
    indice.indexar(doc)

print(f"✅ Índice multimodal criado com {len(indice.documentos)} documentos")
print(f"   - Textos: {sum(1 for d in indice.documentos if d.get('type') == 'text')}")
print(f"   - Imagens: {sum(1 for d in indice.documentos if d.get('type') == 'image')}")

## Etapa 5 — Busca Multimodal

Demonstração do pipeline de query: uma pergunta textual retorna **texto + imagem** simultaneamente.

In [ ]:
from IPython.display import display, Image as IPImage, HTML

def busca_multimodal(query: str, k_texto: int = 2, k_imagem: int = 1) -> dict:
    """
    Executa busca multimodal: retorna texto relevante + imagem relacionada.
    
    Estratégia:
    1. Embedding da query com BGE-M3 → busca textos
    2. Embedding da query com CLIP → busca imagens
    3. Combina resultados
    """
    print(f"\n🔍 Query: \"{query}\"")
    print("=" * 60)
    
    # Embedding para busca de texto
    q_texto_emb = embed_texto(query)
    textos_recuperados = indice.buscar(q_texto_emb, k=k_texto, tipo_filtro="text")
    
    # Embedding CLIP para busca de imagens
    q_imagem_emb = embed_query_clip(query)
    imagens_recuperadas = indice.buscar(q_imagem_emb, k=k_imagem, tipo_filtro="image")
    
    # Exibir resultados de texto
    print(f"\n📄 TEXTOS RECUPERADOS ({len(textos_recuperados)}):")
    for i, doc in enumerate(textos_recuperados, 1):
        print(f"\n[{i}] Score: {doc['_score']:.4f} | Seção: {doc.get('secao', 'N/A')} | Pág. {doc.get('pagina', 'N/A')}")
        conteudo = doc.get('conteudo', '')
        print(f"    {conteudo[:200]}..." if len(conteudo) > 200 else f"    {conteudo}")
    
    # Exibir imagens recuperadas
    print(f"\n🖼️  IMAGENS RECUPERADAS ({len(imagens_recuperadas)}):")
    for i, doc in enumerate(imagens_recuperadas, 1):
        print(f"\n[{i}] Score: {doc['_score']:.4f} | ID: {doc.get('id')}")
        print(f"    Caption: {doc.get('caption', 'N/A')}")
        print(f"    Arquivo: {doc.get('caminho', 'N/A')}")
        
        # Exibir imagem no Colab
        try:
            display(IPImage(doc["caminho"], width=400))
        except Exception as e:
            print(f"    ⚠️ Imagem não disponível para exibição: {e}")
    
    return {
        "textos": textos_recuperados,
        "imagens": imagens_recuperadas
    }


# ============================================================
# QUERY 1: Estrutura organizacional da organização criminosa
# ============================================================
resultado1 = busca_multimodal(
    query="estrutura hierárquica da organização criminosa líderes e operadores",
    k_texto=2,
    k_imagem=1
)

In [ ]:
# ============================================================
# QUERY 2: Materiais apreendidos na operação
# ============================================================
resultado2 = busca_multimodal(
    query="drogas armas apreensões operação policial resultado",
    k_texto=2,
    k_imagem=1
)

In [ ]:
# ============================================================
# QUERY 3: Estatísticas de ocorrências por bairro
# ============================================================
resultado3 = busca_multimodal(
    query="mapa ocorrências criminais zonas bairros incidência",
    k_texto=2,
    k_imagem=1
)

## Etapa 6 — Código de Produção com OpenSearch Real

Este bloco demonstra como o código acima seria implementado com o OpenSearch real.

In [ ]:
# CÓDIGO DE PRODUÇÃO — NÃO EXECUTAR NO COLAB (requer OpenSearch)
# Este bloco é apenas para referência didática

CODIGO_PRODUCAO = '''
from opensearchpy import OpenSearch

client = OpenSearch(
    hosts=[{"host": "localhost", "port": 9200}],
    http_auth=("admin", "senha"),
    use_ssl=True,
    verify_certs=False
)

# 1. Criar índice multimodal
client.indices.create(
    index="multimodal_juridico",
    body={
        "settings": {
            "index": {"knn": True},
            "number_of_shards": 2,
            "number_of_replicas": 1
        },
        "mappings": {
            "properties": {
                "embedding": {
                    "type": "knn_vector",
                    "dimension": 512,
                    "method": {
                        "name": "hnsw",
                        "space_type": "cosinesimil",
                        "engine": "faiss"
                    }
                },
                "tipo": {"type": "keyword"},
                "conteudo": {"type": "text", "analyzer": "portuguese"},
                "caption": {"type": "text"},
                "pagina": {"type": "integer"},
                "secao": {"type": "keyword"}
            }
        }
    }
)

# 2. Indexar documento (texto ou imagem)
def indexar_documento(doc: dict, embedding: list):
    client.index(
        index="multimodal_juridico",
        body={**doc, "embedding": embedding}
    )

# 3. Busca KNN multimodal
def busca_knn(query_embedding: list, tipo: str, k: int = 5):
    return client.search(
        index="multimodal_juridico",
        body={
            "query": {
                "knn": {
                    "embedding": {"vector": query_embedding, "k": k}
                }
            },
            "post_filter": {"term": {"tipo": tipo}}
        }
    )
'''

print("📋 Código de produção (OpenSearch):")
print(CODIGO_PRODUCAO)

## Etapa 7 — Avaliação e Análise

### Métricas de Qualidade

In [ ]:
# Avaliação qualitativa das buscas realizadas

print("=" * 65)
print("RELATÓRIO DE AVALIAÇÃO — Lab 1: Multimodal RAG")
print("=" * 65)

avaliacoes = [
    {
        "query": "estrutura hierárquica da organização criminosa",
        "resultado_texto_esperado": "txt_004 (Análise de Inteligência)",
        "resultado_imagem_esperado": "img_001 (Organograma)",
        "resultado_obtido_texto": resultado1["textos"][0]["id"] if resultado1["textos"] else "N/A",
        "resultado_obtido_imagem": resultado1["imagens"][0]["id"] if resultado1["imagens"] else "N/A",
    },
    {
        "query": "drogas armas apreensões operação",
        "resultado_texto_esperado": "txt_003 (Resultados)",
        "resultado_imagem_esperado": "img_003 (Tabela Apreensões)",
        "resultado_obtido_texto": resultado2["textos"][0]["id"] if resultado2["textos"] else "N/A",
        "resultado_obtido_imagem": resultado2["imagens"][0]["id"] if resultado2["imagens"] else "N/A",
    },
    {
        "query": "mapa ocorrências criminais zonas",
        "resultado_texto_esperado": "txt_005 (Estatísticas)",
        "resultado_imagem_esperado": "img_002 (Mapa de Calor)",
        "resultado_obtido_texto": resultado3["textos"][0]["id"] if resultado3["textos"] else "N/A",
        "resultado_obtido_imagem": resultado3["imagens"][0]["id"] if resultado3["imagens"] else "N/A",
    },
]

acertos_texto = 0
acertos_imagem = 0

for av in avaliacoes:
    acerto_t = av["resultado_obtido_texto"] == av["resultado_texto_esperado"].split(" ")[0]
    acerto_i = av["resultado_obtido_imagem"] == av["resultado_imagem_esperado"].split(" ")[0]
    acertos_texto += int(acerto_t)
    acertos_imagem += int(acerto_i)
    
    print(f"\nQuery: \"{av['query']}\"")
    print(f"  Texto  → Esperado: {av['resultado_texto_esperado']} | Obtido: {av['resultado_obtido_texto']} | {'✅' if acerto_t else '⚠️'}")
    print(f"  Imagem → Esperado: {av['resultado_imagem_esperado']} | Obtido: {av['resultado_obtido_imagem']} | {'✅' if acerto_i else '⚠️'}")

print(f"\n{'='*65}")
print(f"Precisão Texto@1:  {acertos_texto}/{len(avaliacoes)} = {acertos_texto/len(avaliacoes):.1%}")
print(f"Precisão Imagem@1: {acertos_imagem}/{len(avaliacoes)} = {acertos_imagem/len(avaliacoes):.1%}")

print("\n📊 Output esperado: ambas as métricas ≥ 66%")
print("💡 CLIP com textos curtos de caption pode ter precisão variável")
print("   Solução: usar captions mais ricas ou fine-tuning do CLIP no domínio")

## Pergunta de Reflexão

> **Reflexão:** Em um sistema real de inteligência policial, quais são as implicações de usar CLIP (treinado em dados genéricos da internet) para buscar imagens de documentos jurídicos sigilosos? Que estratégias de fine-tuning ou de proteção de dados você adotaria?

---

## Resumo do Lab 1

| Componente | Papel |
|---|---|
| Docling | Extração de texto + imagens do PDF |
| BGE-M3 | Embeddings para textos e tabelas |
| CLIP | Embeddings para imagens |
| OpenSearch | Índice KNN unificado (tipo knn_vector) |
| Query dual | Busca texto com BGE-M3 + imagens com CLIP |

**Próximo:** Lab 2 — Compressão de Contexto com LLMLingua